# 04 – CẢI TIẾN TRÍCH XUẤT THÔNG TIN HÓA ĐƠN SROIE

Notebook này là sản phẩm chính của phần 04. Luật cải tiến nằm trực tiếp trong Notebook (mục 6); Baseline được import từ `src/baseline.py` của nhóm để bảo đảm đúng một nguồn luật.

**Quy tắc:** chỉ dùng Train và Validation, không nạp Test, không sửa Ground Truth. **Chọn luật trên Train, Validation chỉ để xác nhận** (không tinh chỉnh sau khi xem Validation). Mỗi cải tiến là một cờ riêng, được đo riêng (ablation); cờ nào không có lợi trên Train thì bị loại. Chỉ kết luận "tốt hơn" khi số liệu chứng minh.

## 1. Cấu hình và hàm đo

In [1]:
from pathlib import Path
import re, json, itertools
from datetime import datetime
from collections import Counter
import pandas as pd
import numpy as np

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "outputs").exists():
    for c in [PROJECT_DIR.parent, PROJECT_DIR / "project"]:
        if (c / "outputs").exists(): PROJECT_DIR = c; break

OCR_PATH = PROJECT_DIR / "outputs/ocr/train_validation_ocr.csv"
GT_TRAIN = PROJECT_DIR / "outputs/ground_truth/train_ground_truth.csv"
GT_VAL = PROJECT_DIR / "outputs/ground_truth/validation_ground_truth.csv"
BASE_TRAIN = PROJECT_DIR / "outputs/baseline/baseline_train.csv"
BASE_VAL = PROJECT_DIR / "outputs/baseline/baseline_validation.csv"
OUT_DIR = PROJECT_DIR / "outputs/improved"; OUT_DIR.mkdir(parents=True, exist_ok=True)
for p in [OCR_PATH, GT_TRAIN, GT_VAL]:
    assert p.exists(), f"Không tìm thấy file đầu vào: {p}"
BASE_OK = BASE_TRAIN.exists() and BASE_VAL.exists()
if not BASE_OK:
    print("CẢNH BÁO: thiếu baseline_train.csv/baseline_validation.csv của người 3 -> sẽ tự tính baseline từ src/baseline.py, KHÔNG đối chiếu được với file của người 3.")

import sys; sys.path.insert(0, str(PROJECT_DIR))
from src.baseline import extract_fields as extract_baseline, AMOUNT_RE, DATE_RE   # luật Baseline của người 3
assert (PROJECT_DIR / "src" / "improved.py").exists(), "Thiếu src/improved.py (mã phương pháp cải tiến của người 4): git pull hoặc copy file này vào src/"
from src.improved import extract, extract_improved, DEFAULT_CFG, REF_STEPS, OCR_CONFIG, ocr_image, preprocess_image   # luật cải tiến nằm trong src/improved.py

FIELDS = ["company", "date", "address", "total"]

# ----- chuẩn hóa để đánh giá: GIỮ NGUYÊN như 03_baseline để số liệu so sánh được -----
DATE_FORMATS = ["%d/%m/%Y","%d-%m-%Y","%d.%m.%Y","%d/%m/%y","%d-%m-%y","%d.%m.%y","%Y-%m-%d","%Y/%m/%d",
                "%d %b %Y","%d%b%Y","%d-%b-%Y","%d %b %y","%d %B %Y"]
def is_missing(x):
    return x is None or (isinstance(x, float) and pd.isna(x)) or str(x).strip() == ""
def norm_text(s):
    if is_missing(s): return None
    s = re.sub(r"[^\w\s]", " ", str(s).lower()); return re.sub(r"\s+", " ", s).strip()
def norm_total(s):
    if is_missing(s): return None
    nums = re.findall(r"\d[\d,.]*", str(s).replace(" ", ""))
    if not nums: return None
    tok = nums[-1]
    if re.search(r",\d{2}$", tok): tok = tok[:-3] + "." + tok[-2:]
    tok = tok.replace(",", "")
    try: return f"{float(tok):.2f}"
    except ValueError: return None
def norm_date(s):
    if is_missing(s): return None
    core = re.sub(r"\s*([/\-.])\s*", r"\1", str(s).strip())
    for cand in (core, core.replace(" ", "")):
        for f in DATE_FORMATS:
            try: return datetime.strptime(cand, f).strftime("%Y-%m-%d")
            except ValueError: pass
    return norm_text(core)
NORM = {"company": norm_text, "address": norm_text, "date": norm_date, "total": norm_total}
def exact(pred, gold, f):
    if is_missing(pred) or is_missing(gold): return 0
    return int(NORM[f](pred) == NORM[f](gold))

def token_f1(pred, gold):
    # F1 theo token (như 03_baseline) - đề yêu cầu báo F1 trường khi có quy tắc rõ
    p = (norm_text(pred) or "").split(); g = (norm_text(gold) or "").split()
    if not p or not g: return 0.0
    common = sum((Counter(p) & Counter(g)).values())
    if common == 0: return 0.0
    pr, rc = common / len(p), common / len(g); return 2 * pr * rc / (pr + rc)

def gold_in_ocr(f, gold, text):
    """Thông tin đúng có nằm trong chữ OCR không? (cùng định nghĩa với 03_baseline)"""
    if is_missing(gold): return False
    if f == "total": return norm_total(gold) in {norm_total(a) for a in AMOUNT_RE.findall(text)}
    if f == "date": return norm_date(gold) in {norm_date(m.group(0)) for m in DATE_RE.finditer(text)}
    g = set((norm_text(gold) or "").split()); t = set((norm_text(text) or "").split())
    return len(g) > 0 and len(g & t) / len(g) >= 0.8
print("Thư mục dự án:", PROJECT_DIR)

CẢNH BÁO: thiếu baseline_train.csv/baseline_validation.csv của người 3 -> sẽ tự tính baseline từ src/baseline.py, KHÔNG đối chiếu được với file của người 3.
Thư mục dự án: /app


## 2. Nạp và kiểm tra dữ liệu (chỉ Train/Validation)

In [2]:
ocr = pd.read_csv(OCR_PATH, dtype={"key": str}).fillna({"ocr_text": ""})
gt = pd.concat([pd.read_csv(GT_TRAIN, dtype=str).fillna(""), pd.read_csv(GT_VAL, dtype=str).fillna("")], ignore_index=True)
assert ocr.key.is_unique and gt.key.is_unique, "Phát hiện key trùng"
assert set(ocr.split.unique()) <= {"train", "validation"}, "OCR có split ngoài Train/Validation"
assert set(ocr.key) == set(gt.key), "Key OCR và Ground Truth không khớp"
df = ocr.merge(gt[["key"] + FIELDS], on="key", how="inner", validate="one_to_one")
tr, va = df[df.split == "train"].reset_index(drop=True), df[df.split == "validation"].reset_index(drop=True)
print(f"Train={len(tr)}  Validation={len(va)}")

Train=501  Validation=125


## 3. Tái lập baseline và đối chiếu với kết quả của người 3
Chạy lại `src/baseline.py` và **kiểm tra khớp từng ảnh** với `baseline_train.csv` / `baseline_validation.csv`. Nếu không khớp thì dừng, vì mọi so sánh phía sau sẽ vô nghĩa.

In [3]:
def run_baseline(d):
    rows = []
    for _, r in d.iterrows():
        p = extract_baseline(r.ocr_text); row = {"key": r.key, "split": r.split}
        for f in FIELDS:
            row[f + "_pred"] = p.get(f); row[f + "_gold"] = r[f]; row[f + "_em"] = exact(p.get(f), r[f], f)
        rows.append(row)
    return pd.DataFrame(rows)
baseline = run_baseline(df)

if BASE_OK:
    ref = pd.concat([pd.read_csv(BASE_TRAIN, dtype={"key": str}), pd.read_csv(BASE_VAL, dtype={"key": str})]).set_index("key")
    chk = baseline.set_index("key")
    for f in FIELDS:
        diff = int((chk[f + "_em"] != ref.loc[chk.index, f + "_em"]).sum())
        assert diff == 0, f"Baseline tái lập KHÔNG khớp file của người 3 ở trường {f}: {diff} ảnh lệch"
    print("Baseline tái lập khớp 100% với baseline_train/validation.csv của người 3.")
else:
    print("Bỏ qua đối chiếu với file người 3 (thiếu file). Baseline dưới đây tính từ src/baseline.py.")
tbl = pd.DataFrame({s: {**{f: baseline[baseline.split == s][f + "_em"].mean() for f in FIELDS}} for s in ["train", "validation"]})
tbl.loc["MACRO"] = tbl.mean(); tbl.round(4)

Bỏ qua đối chiếu với file người 3 (thiếu file). Baseline dưới đây tính từ src/baseline.py.


,train,validation
company,0.2555,0.248
date,0.6946,0.752
address,0.2535,0.304
total,0.5329,0.568
MACRO,0.4341,0.468


## 4. Phân loại lỗi baseline: do OCR hay do luật?
`rule_error` = chữ đúng **có** trong OCR nhưng luật chọn sai (cải tiến luật sửa được). `ocr_error` = chữ đúng không có trong OCR (luật không sửa được; chỉ tiền xử lý ảnh/OCR khác mới giúp).

In [4]:
def attribution(frame, texts, pred_em_col="_em", gold_col="_gold"):
    out = {}
    for f in FIELDS:
        c = []
        for (_, r), t in zip(frame.iterrows(), texts):
            if r[f + pred_em_col]: c.append("correct")
            elif gold_in_ocr(f, r[f + gold_col], t): c.append("rule_error")
            else: c.append("ocr_error")
        out[f] = pd.Series(c).value_counts()
    return pd.DataFrame(out).fillna(0).astype(int)
bv = baseline[baseline.split == "validation"].reset_index(drop=True)
bt = baseline[baseline.split == "train"].reset_index(drop=True)
print("Baseline – Train"); display(attribution(bt, tr.ocr_text))
print("Baseline – Validation"); display(attribution(bv, va.ocr_text))

Baseline – Train


,company,date,address,total
correct,128,348,127,267
ocr_error,137,126,81,82
rule_error,236,27,293,152


Baseline – Validation


,company,date,address,total
correct,31,94,38,71
ocr_error,27,22,20,14
rule_error,67,9,67,40


In [5]:
# Xem ví dụ lỗi do luật (Train) để thiết kế cải tiến có cơ sở
def show_rule_errors(field, n=3):
    shown = 0
    for (_, r), t in zip(bt.iterrows(), tr.ocr_text):
        if not r[field + "_em"] and gold_in_ocr(field, r[field + "_gold"], t):
            print("=" * 60); print("KEY:", r.key, "| ", field); print("DỰ ĐOÁN:", r[field + "_pred"]); print("ĐÁP ÁN :", r[field + "_gold"])
            print("--- OCR (800 ký tự đầu) ---"); print(t[:800]); shown += 1
            if shown >= n: break
show_rule_errors("total", 2)

KEY: X00016469612 |  total
DỰ ĐOÁN: 91.00
ĐÁP ÁN : 9.00
--- OCR (800 ký tự đầu) ---
tan woon yann
BOOK TA -K (TAMAN DAYA) SDN BHD
789417-W
NO.S: $557 & $8, JALAN SAGU 18,
TAMAN DAYA,
81100 JOHOR BAHRU,
JOHOR
\ F
WAN AAA
hit
Document No : TDO01167104
Date 25/12/2018 8:13:39 PM
Cashier : MANIS
Member :
CASH BILL
CODE/DESC PRICE Disc AMOUIT
QTy Ri RM
9556935040116 KF MODELLING CLAY KIDDY FISH
1 PC * 9.000 6,00 9.00
Total : 91.00
Rour ding Adjustment ; 0.00
Round. :d Total (RM); 9.60
Cash I~ 0.00
CHANGE \00
?.00
GOODS sOLD ARe NOT RE FURNABTEOR
EXCHANGEABLE
THANK YOU
PLEASE COreé AGAR |

KEY: X00016469619 |  total
DỰ ĐOÁN: 60,34
ĐÁP ÁN : 60.30
--- OCR (800 ký tự đầu) ---
tan woon yann
INDAH GIFT & HOME BECO
27, JALAN DEDAF 13,
TAMAN JOHOR JAYA,

81100 JOHOR BAHRU, JOHOR.
Tel:07-3507405 Fax:07-3598140
RECEIPT

19/10/2018 29:49:59 #01
Cashier: CN Location/SP: 05 /O531
HB: 4026588
Room No: O1 050100025279
Desc/Tten Qty Price Amt(Ri)
ST-PRIVILEGE CARD/GD INDAH
88888 1 10.00 10.00
GF-TABLE LAMP

## 5. Tiền xử lý ảnh (cần ảnh trong file parquet)
Theo `02_ocr.ipynb`, OCR CSV của người 1 được tạo bằng **Tesseract 5.5 (`--oem 3 --psm 6`, `eng`)** trên ảnh **grayscale + phóng 2x** (đó là mốc đối chứng ở đây). Người 1 thử nhanh trên **một** ảnh mẫu và thấy nhị phân Otsu **kém hơn nhiều** (WER 0.69 so với 0.35; độ tin cậy 51 so với 70), nên mục này đo lại trên nhiều ảnh các biến thể: `resize3x`, `deskew`, `clahe`, `median`, `otsu`, `adaptive`.

Cách đo: OCR lại một tập con ngẫu nhiên (`MAX_IMAGES` ảnh, `seed=42`) rồi so **tỉ lệ chữ đúng có trong OCR** (`gold_in_ocr`) với OCR gốc của người 1 trên đúng các ảnh đó. Ảnh đọc từ file Train parquet (file test **không** được đọc). Tốc độ khoảng 3 giây/ảnh/biến thể (người 1 mất ~32 phút cho 626 ảnh).
Quy trình: (1) đặt `RUN_PREPROC = True`, chạy so sánh trên Train; (2) nếu có biến thể thắng rõ, điền tên vào `FULL_OCR_VARIANT` để OCR lại toàn bộ 626 ảnh; (3) notebook chỉ chuyển sang OCR mới nếu Validation cũng tăng.

In [6]:
RUN_PREPROC = False
PARQUET_TRAIN = PROJECT_DIR / "data" / "ICDAR2019-SROIE" / "data" / "train-00000-of-00001.parquet"   # như 02_ocr; KHÔNG đọc file test
PREPROC_EVAL_SPLIT = "train"      # chọn biến thể trên Train; đổi "validation" để xác nhận
MAX_IMAGES = 60                   # ~3 phút/biến thể; None = cả split
VARIANTS = {"resize3x": ["gray", "resize3"], "deskew": ["gray", "deskew", "resize2"], "clahe": ["gray", "resize2", "clahe"],
            "median": ["gray", "resize2", "median"], "otsu": ["gray", "resize2", "otsu"], "adaptive": ["gray", "resize2", "adaptive"]}
OCR_ORIG = dict(zip(df.key, df.ocr_text))     # OCR gốc của người 1 (mốc đối chứng, không bị ghi đè)

def _iter_images(keys):
    # đọc từng lô từ parquet Train, chỉ lấy các key cần (không nạp hết 626 ảnh vào RAM)
    import pyarrow.parquet as pq
    from PIL import Image
    from io import BytesIO
    keys = set(keys)
    for batch in pq.ParquetFile(PARQUET_TRAIN).iter_batches(batch_size=16, columns=["key", "image"]):
        for r in batch.to_pylist():
            if str(r["key"]) in keys:
                yield str(r["key"]), Image.open(BytesIO(r["image"]["bytes"])).convert("RGB")

def recall_table(frame, texts):
    return {f: float(np.mean([gold_in_ocr(f, g, texts[k]) for k, g in zip(frame.key, frame[f])])) for f in FIELDS}

if RUN_PREPROC:
    import time
    assert PARQUET_TRAIN.exists(), f"Không thấy {PARQUET_TRAIN}: sửa PARQUET_TRAIN cho đúng thư mục data của dự án"
    d = tr if PREPROC_EVAL_SPLIT == "train" else va
    sub = d.sample(n=min(MAX_IMAGES or len(d), len(d)), random_state=42).reset_index(drop=True)
    imgs = dict(_iter_images(sub.key))
    assert set(imgs) == set(sub.key), "Thiếu ảnh trong parquet"
    # kiểm tra tái tạo: OCR lại bằng đúng cấu hình của người 1 có ra đúng văn bản trong CSV không?
    same = sum(ocr_image(imgs[k], REF_STEPS).strip() == OCR_ORIG[k].strip() for k in list(imgs)[:3])
    print(f"Tái tạo OCR gốc: {same}/3 ảnh giống hệt CSV của người 1 (nếu 0/3, phiên bản Tesseract/OpenCV khác -> so sánh dưới đây vẫn dùng văn bản CSV làm mốc nhưng chênh lệch có thể do phiên bản).")
    rec = {"control: gray+2x (CSV người 1)": recall_table(sub, OCR_ORIG)}
    rec["control: gray+2x (CSV người 1)"]["giay"] = 0
    for name, steps in VARIANTS.items():
        t0 = time.time(); txt = {k: ocr_image(img, steps) for k, img in imgs.items()}
        rec[name] = {**recall_table(sub, txt), "giay": round(time.time() - t0)}
        pd.DataFrame({"key": list(txt), "ocr_text": list(txt.values())}).to_csv(OUT_DIR / f"ocr_subset_{name}.csv", index=False, encoding="utf-8-sig")
    pre = pd.DataFrame(rec).T; pre["mean"] = pre[FIELDS].mean(axis=1)
    pre["delta_vs_control"] = pre["mean"] - pre.loc["control: gray+2x (CSV người 1)", "mean"]
    print(f"Tỉ lệ chữ đúng có trong OCR, {PREPROC_EVAL_SPLIT}, n={len(sub)}:"); display(pre.round(4))
    pre.round(4).to_csv(OUT_DIR / "tien_xu_ly_anh_so_sanh.csv", encoding="utf-8-sig")
else:
    print("Bỏ qua tiền xử lý ảnh (RUN_PREPROC=False) – chưa có kết quả, KHÔNG được kết luận về grayscale/resize/threshold/deskew.")

Bỏ qua tiền xử lý ảnh (RUN_PREPROC=False) – chưa có kết quả, KHÔNG được kết luận về grayscale/resize/threshold/deskew.


In [7]:
# OCR lại TOÀN BỘ 626 ảnh bằng biến thể thắng (~30 phút). Điền tên biến thể vào đây sau khi xem bảng ở trên.
FULL_OCR_VARIANT = None           # ví dụ "deskew"
FULL_PATH = OUT_DIR / f"ocr_full_{FULL_OCR_VARIANT}.csv" if FULL_OCR_VARIANT else None
if FULL_OCR_VARIANT:
    import time
    steps = VARIANTS[FULL_OCR_VARIANT]
    done = pd.read_csv(FULL_PATH, dtype={"key": str}).fillna("") if FULL_PATH.exists() else pd.DataFrame(columns=["key", "ocr_text"])
    todo = [k for k in df.key if k not in set(done.key)]; rows = done.to_dict("records"); t0 = time.time()   # chạy lại được nếu bị ngắt
    for i, (k, img) in enumerate(_iter_images(todo), 1):
        rows.append({"key": k, "ocr_text": ocr_image(img, steps)})
        if i % 25 == 0 or i == len(todo):
            pd.DataFrame(rows).to_csv(FULL_PATH, index=False, encoding="utf-8-sig"); print(f"{i}/{len(todo)}  {round(time.time() - t0)}s")
    pd.DataFrame(rows).to_csv(FULL_PATH, index=False, encoding="utf-8-sig")

# Chỉ chuyển sang OCR mới khi Validation cũng tăng (so với OCR gốc của người 1)
OCR_SOURCE, PREPROC_STEPS = "original (gray+resize2x)", REF_STEPS
if FULL_PATH is not None and FULL_PATH.exists():
    new = pd.read_csv(FULL_PATH, dtype={"key": str}).fillna("").set_index("key")["ocr_text"].to_dict()
    if set(df.key) <= set(new):
        r_old, r_new = recall_table(va, OCR_ORIG), recall_table(va, new)
        cmp_ocr = pd.DataFrame({"gốc": r_old, FULL_OCR_VARIANT: r_new}).T; cmp_ocr["mean"] = cmp_ocr[FIELDS].mean(axis=1); display(cmp_ocr.round(4))
        if cmp_ocr.loc[FULL_OCR_VARIANT, "mean"] > cmp_ocr.loc["gốc", "mean"] + 0.005:
            df["ocr_text"] = df.key.map(new); tr, va = df[df.split == "train"].reset_index(drop=True), df[df.split == "validation"].reset_index(drop=True)
            OCR_SOURCE, PREPROC_STEPS = FULL_OCR_VARIANT, VARIANTS[FULL_OCR_VARIANT]
            print("ĐÃ chuyển sang OCR mới:", OCR_SOURCE)
        else:
            print("OCR mới KHÔNG tốt hơn rõ rệt trên Validation -> giữ OCR gốc.")
    else:
        print("File OCR đầy đủ chưa phủ hết key -> giữ OCR gốc.")
print("Nguồn OCR dùng cho các mục sau:", OCR_SOURCE)

Nguồn OCR dùng cho các mục sau: original (gray+resize2x)


## 6. Luật cải tiến (mỗi cải tiến là một cờ)

**Nguyên tắc: khi mọi cờ tắt, kết quả GIỐNG HỆT baseline của người 3** (luật baseline được sao chép nguyên văn vào `src/improved.py` và có `assert` kiểm tra từng ảnh). Nhờ vậy mọi mức tăng/giảm trong ablation là so với baseline thật.

| Cờ | Cải tiến | Lỗi nhắm tới (từ ghi chú của người 3) |
|---|---|---|
| `clean_text` | bỏ ký tự rác đầu dòng, `\x0c`, dòng chỉ có ký hiệu (áp cho mọi trường) | OCR nhiễu |
| `clean_date` | sửa nhầm O/0, l/1, S/5… trong chuỗi ngày | date: lỗi đọc chữ |
| `date_v2` | kiểm tra ngày hợp lệ (loại `4 - 6/300`), nhiều định dạng, ưu tiên dòng `DATE`, bỏ ngày hết hạn; không tìm được thì dùng luật baseline | date sai/None |
| `clean_amount` | sửa nhầm ký tự trong số tiền | total: `9.O0` |
| `total_kw` | thêm từ khóa (`NETT TOTAL`, `TOTAL INCL`, `TOTAL DUE`…) và không loại nhầm `TOTAL INCL. GST` | total chọn sai dòng/None |
| `total_nospace` | so khớp từ khóa sau khi bỏ khoảng trắng (`NetTotal`, `GRANDTOTAL`) | total None do từ khóa dính liền |
| `total_gstcut` | bỏ phần `GST/TAX SUMMARY` khi tìm total | lấy nhầm số trong bảng GST |
| `total_pos` | không có từ khóa → số lớn nhất ở phần dưới hóa đơn | total None |
| `company_v2` | chấm điểm dòng đầu (hậu tố SDN BHD…, chữ HOA, loại TEL/INVOICE/địa chỉ, phạt dòng toàn chữ thường như tên người), ghép `SDN BHD` bị tách dòng | company = dòng 0 là tên người |
| `addr_v2` | dấu hiệu địa chỉ rộng hơn, cắt phần `TEL/FAX` cuối dòng | địa chỉ thiếu/thừa |
| `addr_merge` | lấy **khối dòng liên tiếp** (cho phép hỏng 1 dòng) thay vì gom rời rạc | địa chỉ thừa dòng |

Quan trọng: address luôn được tính **từ vị trí dòng company đã chọn** (kể cả khi tắt mọi cờ address, vẫn dùng luật baseline nhưng bắt đầu từ dòng company mới), nên `company_v2` đúng thì address cũng hết lẫn tên công ty.

Mã luật cải tiến **không viết trong notebook** mà nằm ở `src/improved.py` (notebook import ở cell 1), để người 5 và `demo.py` dùng lại cùng một nguồn. Khi mọi cờ tắt, `extract(text, DEFAULT_CFG)` cho kết quả giống hệt baseline; hàm `extract_improved(ocr_text)` (dùng cho demo) tự đọc `outputs/improved/frozen_config.json`.

In [8]:
# Kiểm tra nhanh luật cải tiến
demo = "tan woon yann\nABC TRADING SDN BHD\nNO 1 JALAN BUNGA\n50000 KUALA LUMPUR\nTEL: 012\nDATE: 25/12/2018\nSUBTOTAL 8.00\nNetTotal 9.00\nGST SUMMARY\nSR 6% 0.50 TOTAL 9.50"
full = {k: True for k in DEFAULT_CFG}
print(extract(demo, full)); assert extract(demo, full)["total"] == "9.00" and extract(demo, full)["company"] == "ABC TRADING SDN BHD"
print("OK")

# BẮT BUỘC: mọi cờ tắt phải cho kết quả giống hệt baseline của người 3, từng ảnh
mism = [k for k, t in zip(df.key, df.ocr_text) if extract(t, DEFAULT_CFG) != extract_baseline(t)]
assert not mism, f"Luật mọi-cờ-tắt KHÁC baseline ở {len(mism)} ảnh, ví dụ: {mism[:3]}"
print("Mọi cờ tắt == baseline của người 3 trên", len(df), "ảnh.")

{'company': 'ABC TRADING SDN BHD', 'date': '25/12/2018', 'address': 'NO 1 JALAN BUNGA 50000 KUALA LUMPUR', 'total': '9.00'}
OK
Mọi cờ tắt == baseline của người 3 trên 626 ảnh.


## 7. Ablation – từng cải tiến đo riêng
Mốc là **baseline** (mọi cờ tắt = đúng luật của người 3). Mỗi cờ được bật **một mình** rồi đo cả 4 trường; cột `Δ…` là chênh lệch so với baseline. Một số cờ chỉ có tác dụng khi đi cùng cờ khác (ví dụ `addr_merge` cần dòng company đúng), nên có thêm bảng **tích lũy** theo thứ tự. Cột `Train` dùng để chọn; `Validation` chỉ để xem/xác nhận.

In [9]:
def predict(d, cfg):
    return pd.DataFrame([extract(t, cfg) for t in d["ocr_text"]], index=d.index)
def evaluate(d, cfg):
    p = predict(d, cfg)
    e = {f: float(np.mean([exact(a, b, f) for a, b in zip(p[f], d[f])])) for f in FIELDS}
    e["macro"] = float(np.mean([e[f] for f in FIELDS])); return e
COLS = FIELDS + ["macro"]
FLAG_FIELD = {"clean_text":"macro","clean_date":"date","date_v2":"date","clean_amount":"total","total_kw":"total",
              "total_nospace":"total","total_gstcut":"total","total_pos":"total","company_v2":"company",
              "addr_v2":"address","addr_merge":"address"}
BASE = "baseline (mọi cờ tắt)"
rows = []
def add(name, cfg):
    for sp, d in [("Train", tr), ("Validation", va)]:
        e = evaluate(d, cfg); rows.append(dict(config=name, split=sp, **{c: round(e[c], 4) for c in COLS}))
add(BASE, DEFAULT_CFG)
for flag in DEFAULT_CFG: add(f"+ {flag}", {**DEFAULT_CFG, flag: True})
add("tất cả cờ bật", {k: True for k in DEFAULT_CFG})
abl = pd.DataFrame(rows)
def show_abl(table):
    for sp in ["Train", "Validation"]:
        a = table[table.split == sp].set_index("config").drop(columns="split")
        d = (a[COLS] - a.loc[a.index[0], COLS]).round(4).add_prefix("Δ")
        print(f"--- {sp} ---"); display(pd.concat([a, d], axis=1))
print("Nguồn OCR:", OCR_SOURCE, "(nếu khác 'original' thì dòng baseline ở đây là LUẬT baseline trên OCR mới; mục 9 so với baseline gốc của người 3)")
print("ABLATION TỪNG CỜ (bật một mình):"); show_abl(abl)
abl.to_csv(OUT_DIR / "ablation_tung_co.csv", index=False, encoding="utf-8-sig")

# Tích lũy theo thứ tự (cờ phụ thuộc nhau mới lộ tác dụng)
ORDER = ["clean_text","clean_date","date_v2","clean_amount","total_kw","total_nospace","total_gstcut","total_pos",
         "company_v2","addr_v2","addr_merge"]
rows2, cur = [], dict(DEFAULT_CFG)
for sp, d in [("Train", tr), ("Validation", va)]:
    e = evaluate(d, cur); rows2.append(dict(config=BASE, split=sp, **{c: round(e[c], 4) for c in COLS}))
for flag in ORDER:
    cur[flag] = True
    for sp, d in [("Train", tr), ("Validation", va)]:
        e = evaluate(d, cur); rows2.append(dict(config=f"+ {flag} (tích lũy)", split=sp, **{c: round(e[c], 4) for c in COLS}))
abl_cum = pd.DataFrame(rows2)
print("ABLATION TÍCH LŨY:"); show_abl(abl_cum)
abl_cum.to_csv(OUT_DIR / "ablation_tich_luy.csv", index=False, encoding="utf-8-sig")

Nguồn OCR: original (gray+resize2x) (nếu khác 'original' thì dòng baseline ở đây là LUẬT baseline trên OCR mới; mục 9 so với baseline gốc của người 3)
ABLATION TỪNG CỜ (bật một mình):
--- Train ---


,company,date,address,total,macro,Δcompany,Δdate,Δaddress,Δtotal,Δmacro
config,,,,,,,,,,
baseline (mọi cờ tắt),0.2555,0.6946,0.2535,0.5329,0.4341,0.0000,0.0000,0.0000,0.0000,0.0000
+ clean_text,0.2595,0.6946,0.2555,0.5329,0.4356,0.0040,0.0000,0.0020,0.0000,0.0015
+ clean_date,0.2555,0.6966,0.2535,0.5329,0.4346,0.0000,0.0020,0.0000,0.0000,0.0005
+ date_v2,0.2555,0.7425,0.2535,0.5329,0.4461,0.0000,0.0479,0.0000,0.0000,0.0120
+ clean_amount,0.2555,0.6946,0.2535,0.5349,0.4346,0.0000,0.0000,0.0000,0.0020,0.0005
+ total_kw,0.2555,0.6946,0.2535,0.6088,0.4531,0.0000,0.0000,0.0000,0.0759,0.0190
+ total_nospace,0.2555,0.6946,0.2535,0.5329,0.4341,0.0000,0.0000,0.0000,0.0000,0.0000
+ total_gstcut,0.2555,0.6946,0.2535,0.5329,0.4341,0.0000,0.0000,0.0000,0.0000,0.0000
+ total_pos,0.2555,0.6946,0.2535,0.5768,0.4451,0.0000,0.0000,0.0000,0.0439,0.0110


--- Validation ---


,company,date,address,total,macro,Δcompany,Δdate,Δaddress,Δtotal,Δmacro
config,,,,,,,,,,
baseline (mọi cờ tắt),0.248,0.752,0.304,0.568,0.468,0.000,0.000,0.000,0.000,0.000
+ clean_text,0.256,0.752,0.304,0.568,0.470,0.008,0.000,0.000,0.000,0.002
+ clean_date,0.248,0.752,0.304,0.568,0.468,0.000,0.000,0.000,0.000,0.000
+ date_v2,0.248,0.808,0.304,0.568,0.482,0.000,0.056,0.000,0.000,0.014
+ clean_amount,0.248,0.752,0.304,0.568,0.468,0.000,0.000,0.000,0.000,0.000
+ total_kw,0.248,0.752,0.304,0.664,0.492,0.000,0.000,0.000,0.096,0.024
+ total_nospace,0.248,0.752,0.304,0.576,0.470,0.000,0.000,0.000,0.008,0.002
+ total_gstcut,0.248,0.752,0.304,0.568,0.468,0.000,0.000,0.000,0.000,0.000
+ total_pos,0.248,0.752,0.304,0.600,0.476,0.000,0.000,0.000,0.032,0.008


ABLATION TÍCH LŨY:
--- Train ---


,company,date,address,total,macro,Δcompany,Δdate,Δaddress,Δtotal,Δmacro
config,,,,,,,,,,
baseline (mọi cờ tắt),0.2555,0.6946,0.2535,0.5329,0.4341,0.0000,0.0000,0.0000,0.0000,0.0000
+ clean_text (tích lũy),0.2595,0.6946,0.2555,0.5329,0.4356,0.0040,0.0000,0.0020,0.0000,0.0015
+ clean_date (tích lũy),0.2595,0.6966,0.2555,0.5329,0.4361,0.0040,0.0020,0.0020,0.0000,0.0020
+ date_v2 (tích lũy),0.2595,0.7465,0.2555,0.5329,0.4486,0.0040,0.0519,0.0020,0.0000,0.0145
+ clean_amount (tích lũy),0.2595,0.7465,0.2555,0.5349,0.4491,0.0040,0.0519,0.0020,0.0020,0.0150
+ total_kw (tích lũy),0.2595,0.7465,0.2555,0.6108,0.4681,0.0040,0.0519,0.0020,0.0779,0.0340
+ total_nospace (tích lũy),0.2595,0.7465,0.2555,0.6108,0.4681,0.0040,0.0519,0.0020,0.0779,0.0340
+ total_gstcut (tích lũy),0.2595,0.7465,0.2555,0.6108,0.4681,0.0040,0.0519,0.0020,0.0779,0.0340
+ total_pos (tích lũy),0.2595,0.7465,0.2555,0.6547,0.4790,0.0040,0.0519,0.0020,0.1218,0.0449


--- Validation ---


,company,date,address,total,macro,Δcompany,Δdate,Δaddress,Δtotal,Δmacro
config,,,,,,,,,,
baseline (mọi cờ tắt),0.248,0.752,0.304,0.568,0.468,0.000,0.000,0.000,0.000,0.000
+ clean_text (tích lũy),0.256,0.752,0.304,0.568,0.470,0.008,0.000,0.000,0.000,0.002
+ clean_date (tích lũy),0.256,0.752,0.304,0.568,0.470,0.008,0.000,0.000,0.000,0.002
+ date_v2 (tích lũy),0.256,0.808,0.304,0.568,0.484,0.008,0.056,0.000,0.000,0.016
+ clean_amount (tích lũy),0.256,0.808,0.304,0.568,0.484,0.008,0.056,0.000,0.000,0.016
+ total_kw (tích lũy),0.256,0.808,0.304,0.664,0.508,0.008,0.056,0.000,0.096,0.040
+ total_nospace (tích lũy),0.256,0.808,0.304,0.672,0.510,0.008,0.056,0.000,0.104,0.042
+ total_gstcut (tích lũy),0.256,0.808,0.304,0.672,0.510,0.008,0.056,0.000,0.104,0.042
+ total_pos (tích lũy),0.256,0.808,0.304,0.704,0.518,0.008,0.056,0.000,0.136,0.050


## 8. Chọn phương pháp trên **Train** và đóng băng cấu hình
Duyệt mọi tổ hợp cờ theo nhóm **trên Train**: `clean_text` (đo bằng macro 4 trường) → date → total → company → address (address dùng company đã chọn). Mốc so sánh là baseline (mọi cờ tắt). Chọn EM Train cao nhất; hòa thì chọn ít cờ hơn; chỉ bật khi hơn "tắt hết" ít nhất `MIN_GAIN` (0.5 điểm phần trăm). Validation **không** tham gia chọn.

In [10]:
MIN_GAIN = 0.005
SELECT_ON = "train"   # "train" (khuyên dùng: Validation chỉ để xác nhận) hoặc "validation"
assert SELECT_ON in ("train", "validation")
SEL = tr if SELECT_ON == "train" else va
GROUPS = [("macro", ["clean_text"]), ("date", ["clean_date","date_v2"]), ("total", ["clean_amount","total_kw","total_nospace","total_gstcut","total_pos"]),
          ("company", ["company_v2"]), ("address", ["addr_v2","addr_merge"])]
chosen, log = dict(DEFAULT_CFG), []
for field, flags in GROUPS:
    results = []
    for combo in itertools.product([False, True], repeat=len(flags)):
        cfg = {**chosen, **dict(zip(flags, combo))}
        em = evaluate(SEL, cfg)[field]; results.append((em, -sum(combo), combo))
        log.append(dict(field=field, **dict(zip(flags, combo)), EM_chon=round(em, 4)))
    best = max(results); off = next(r for r in results if not any(r[2]))
    use = best[2] if best[0] >= off[0] + MIN_GAIN else off[2]
    chosen.update(dict(zip(flags, use)))
    print(f"{field:8s}: baseline={off[0]:.4f} -> chọn {dict(zip(flags, use))}  EM_chon={max(r for r in results if r[2]==use)[0]:.4f}")
pd.DataFrame(log).to_csv(OUT_DIR / "tim_kiem_to_hop_co_train.csv", index=False)
FROZEN = dict(cfg=chosen, selected_on=SELECT_ON, min_gain=MIN_GAIN, n_train=len(tr), n_validation=len(va), ocr_source=OCR_SOURCE,
              ocr=dict(steps=PREPROC_STEPS, tesseract_config=OCR_CONFIG, lang="eng"))
(OUT_DIR / "frozen_config.json").write_text(json.dumps(FROZEN, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(chosen, indent=1))

macro   : baseline=0.4341 -> chọn {'clean_text': False}  EM_chon=0.4341
date    : baseline=0.6946 -> chọn {'clean_date': True, 'date_v2': True}  EM_chon=0.7465
total   : baseline=0.5329 -> chọn {'clean_amount': True, 'total_kw': True, 'total_nospace': False, 'total_gstcut': False, 'total_pos': True}  EM_chon=0.6567
company : baseline=0.2555 -> chọn {'company_v2': True}  EM_chon=0.3453
address : baseline=0.2575 -> chọn {'addr_v2': True, 'addr_merge': True}  EM_chon=0.2774
{
 "clean_text": false,
 "clean_date": true,
 "date_v2": true,
 "clean_amount": true,
 "total_kw": true,
 "total_nospace": false,
 "total_gstcut": false,
 "total_pos": true,
 "company_v2": true,
 "addr_v2": true,
 "addr_merge": true
}


In [11]:
# Từng cờ trong cấu hình cuối đóng góp bao nhiêu (tắt từng cờ), trên Train và Validation
fin_tr, fin_va = evaluate(tr, chosen), evaluate(va, chosen)
loo = []
for flag, on in chosen.items():
    if on:
        f = FLAG_FIELD[flag]; etr, eva = evaluate(tr, {**chosen, flag: False}), evaluate(va, {**chosen, flag: False})
        loo.append(dict(co=flag, truong=f, dong_gop_train=round(fin_tr[f] - etr[f], 4), dong_gop_validation=round(fin_va[f] - eva[f], 4)))
loo = pd.DataFrame(loo); display(loo) if len(loo) else print("Không có cờ nào được bật.")

,co,truong,dong_gop_train,dong_gop_validation
0,clean_date,date,0.0040,0.000
1,date_v2,date,0.0499,0.056
2,clean_amount,total,0.0020,0.000
3,total_kw,total,0.0778,0.096
4,total_pos,total,0.0459,0.032
5,company_v2,company,0.0898,0.112
6,addr_v2,address,0.0060,0.008
7,addr_merge,address,0.0140,0.024


## 9. So sánh baseline và cải tiến trên cùng Validation (đã đóng băng)

In [12]:
pv, pt = predict(va, chosen), predict(tr, chosen)
def compare(d, p, b):
    rows = []
    for f in FIELDS:
        imp = np.array([exact(a, g, f) for a, g in zip(p[f], d[f])]); bas = b[f + "_em"].to_numpy()
        rows.append(dict(field=f, baseline=bas.mean(), cai_tien=imp.mean(), chenh_lech=imp.mean() - bas.mean(),
                         sua_dung=int(((bas == 0) & (imp == 1)).sum()), lam_hong=int(((bas == 1) & (imp == 0)).sum())))
    c = pd.DataFrame(rows).set_index("field")
    c.loc["MACRO"] = [c.baseline.mean(), c.cai_tien.mean(), c.chenh_lech.mean(), c.sua_dung.sum(), c.lam_hong.sum()]
    return c.round(4)
cmp_val, cmp_tr = compare(va, pv, bv), compare(tr, pt, bt)
print("VALIDATION (n=%d)" % len(va)); display(cmp_val)
print("TRAIN (n=%d)" % len(tr)); display(cmp_tr)
cmp_val.assign(split="validation").reset_index().to_csv(OUT_DIR / "validation_and_train_comparison.csv", index=False, encoding="utf-8-sig")
pd.concat([cmp_tr.assign(split="train"), cmp_val.assign(split="validation")]).reset_index().to_csv(OUT_DIR / "validation_and_train_comparison.csv", index=False, encoding="utf-8-sig")

VALIDATION (n=125)


,baseline,cai_tien,chenh_lech,sua_dung,lam_hong
field,,,,,
company,0.248,0.360,0.112,14.0,0.0
date,0.752,0.808,0.056,7.0,0.0
address,0.304,0.352,0.048,6.0,0.0
total,0.568,0.696,0.128,16.0,0.0
MACRO,0.468,0.554,0.086,43.0,0.0


TRAIN (n=501)


,baseline,cai_tien,chenh_lech,sua_dung,lam_hong
field,,,,,
company,0.2555,0.3453,0.0898,46.0,1.0
date,0.6946,0.7465,0.0519,28.0,2.0
address,0.2535,0.2774,0.0240,12.0,0.0
total,0.5329,0.6567,0.1238,66.0,4.0
MACRO,0.4341,0.5065,0.0724,152.0,7.0


In [13]:
# F1 token cho company/address (đề yêu cầu báo F1 trường khi có quy tắc rõ)
f1 = []
for sp, d, p, b in [("train", tr, pt, bt), ("validation", va, pv, bv)]:
    for f in ["company", "address"]:
        fb = np.mean([token_f1(x, g) for x, g in zip(b[f + "_pred"], d[f])]); fi = np.mean([token_f1(x, g) for x, g in zip(p[f], d[f])])
        f1.append(dict(split=sp, field=f, baseline_F1=round(fb, 4), cai_tien_F1=round(fi, 4), chenh_lech=round(fi - fb, 4)))
f1 = pd.DataFrame(f1); display(f1); f1.to_csv(OUT_DIR / "f1_company_address.csv", index=False, encoding="utf-8-sig")

,split,field,baseline_F1,cai_tien_F1,chenh_lech
0,train,company,0.5143,0.7124,0.1981
1,train,address,0.7003,0.7105,0.0102
2,validation,company,0.4980,0.7426,0.2447
3,validation,address,0.7567,0.7819,0.0252


In [14]:
# Các ca baseline ĐÚNG nhưng cải tiến SAI (bị làm hỏng) – xem để chỉnh luật hoặc gửi cho người hỗ trợ
def show_regressions(field, d=va, p=pv, b=bv, n=3):
    shown = 0
    for i in range(len(d)):
        if b.at[i, field + "_em"] == 1 and exact(p.at[i, field], d.at[i, field], field) == 0:
            print("=" * 60); print("KEY:", d.at[i, "key"], "|", field)
            print("BASELINE:", b.at[i, field + "_pred"]); print("CẢI TIẾN:", p.at[i, field]); print("ĐÁP ÁN  :", d.at[i, field])
            print("--- OCR (700 ký tự đầu) ---"); print(d.at[i, "ocr_text"][:700]); shown += 1
            if shown >= n: break
    if not shown: print(field, ": không có ca nào bị làm hỏng.")
for f in FIELDS: print("#" * 20, f, "- bị làm hỏng:", int(((bv[f + "_em"] == 1) & (np.array([exact(a, g, f) for a, g in zip(pv[f], va[f])]) == 0)).sum()))
# show_regressions("address"); show_regressions("total")   # bỏ dấu # để xem ví dụ

#################### company - bị làm hỏng: 0
#################### date - bị làm hỏng: 0
#################### address - bị làm hỏng: 0
#################### total - bị làm hỏng: 0


In [15]:
# Sau cải tiến còn bao nhiêu lỗi do luật / do OCR (Validation)
iv = va[["key"]].copy()
for f in FIELDS: iv[f + "_em"] = [exact(a, g, f) for a, g in zip(pv[f], va[f])]; iv[f + "_gold"] = va[f].values
print("Cải tiến – Validation"); display(attribution(iv, va.ocr_text))

Cải tiến – Validation


,company,date,address,total
correct,45,101,44,87
ocr_error,27,22,20,14
rule_error,53,2,61,24


## 10. Kết luận theo số liệu

In [16]:
g = cmp_val.loc["MACRO"]
if g.chenh_lech > 0 and g.sua_dung > g.lam_hong:
    verdict = f"Cải tiến tốt hơn baseline trên Validation: Macro EM {g.baseline:.3f} -> {g.cai_tien:.3f} ({g.chenh_lech:+.3f}); sửa đúng {int(g.sua_dung)}, làm hỏng {int(g.lam_hong)} lượt trường/ảnh."
elif g.chenh_lech == 0 and g.sua_dung == g.lam_hong:
    verdict = "Không có khác biệt trên Validation: KHÔNG tuyên bố cải tiến tốt hơn baseline."
else:
    verdict = f"Cải tiến KHÔNG tốt hơn baseline trên Validation (chênh lệch {g.chenh_lech:+.3f}; sửa {int(g.sua_dung)}, hỏng {int(g.lam_hong)})."
print(verdict)
for f in FIELDS: print(f"  {f:8s}: {cmp_val.loc[f,'baseline']:.3f} -> {cmp_val.loc[f,'cai_tien']:.3f} ({cmp_val.loc[f,'chenh_lech']:+.3f})")
print("Lưu ý: Validation chỉ %d ảnh nên chênh lệch vài ảnh chưa kết luận mạnh; cần đối chiếu thêm với Train (mục 9)." % len(va))

Cải tiến tốt hơn baseline trên Validation: Macro EM 0.468 -> 0.554 (+0.086); sửa đúng 43, làm hỏng 0 lượt trường/ảnh.
  company : 0.248 -> 0.360 (+0.112)
  date    : 0.752 -> 0.808 (+0.056)
  address : 0.304 -> 0.352 (+0.048)
  total   : 0.568 -> 0.696 (+0.128)
Lưu ý: Validation chỉ 125 ảnh nên chênh lệch vài ảnh chưa kết luận mạnh; cần đối chiếu thêm với Train (mục 9).


## 11. Xuất file và bàn giao cho người 5

In [17]:
rows = []
for d, p in [(tr, pt), (va, pv)]:
    for i in d.index:
        r = {"key": d.at[i, "key"], "split": d.at[i, "split"], "ocr_text_raw": d.at[i, "ocr_text"]}
        for f in FIELDS:
            r[f + "_pred_raw"] = p.at[i, f]; r[f + "_pred_norm"] = NORM[f](p.at[i, f])
            r[f + "_gold"] = d.at[i, f]; r[f + "_gold_norm"] = NORM[f](d.at[i, f]); r[f + "_em"] = exact(p.at[i, f], d.at[i, f], f)
            if f in ("company", "address"): r[f + "_f1"] = token_f1(p.at[i, f], d.at[i, f])
        rows.append(r)
improved = pd.DataFrame(rows)
for sp, name in [("train", "improved_train.csv"), ("validation", "improved_validation.csv")]:
    improved[improved.split == sp].to_csv(OUT_DIR / name, index=False, encoding="utf-8-sig")

DESC = {"clean_date":"Sửa nhầm ký tự O/0, l/1, S/5 trong chuỗi ngày","date_v2":"Kiểm tra ngày hợp lệ, nhiều định dạng, ưu tiên dòng DATE, bỏ ngày hết hạn",
 "clean_amount":"Sửa nhầm ký tự trong số tiền","total_kw":"Thêm từ khóa total, không loại nhầm TOTAL INCL. GST",
 "total_nospace":"So khớp từ khóa bỏ khoảng trắng (NetTotal)","total_gstcut":"Bỏ phần GST/TAX SUMMARY khi tìm total",
 "total_pos":"Không có từ khóa → số lớn nhất ở phần dưới hóa đơn","clean_text":"Làm sạch ký tự rác/dòng ký hiệu (đo bằng macro)",
 "company_v2":"Chấm điểm dòng tên công ty (hậu tố, chữ HOA, loại nhiễu), ghép SDN BHD bị tách dòng",
 "addr_v2":"Dấu hiệu địa chỉ rộng hơn, cắt TEL/FAX cuối dòng","addr_merge":"Lấy khối dòng địa chỉ liên tiếp (cho phép hỏng 1 dòng)"}
A = abl.pivot(index="config", columns="split")
tb = []
for flag in DEFAULT_CFG:
    f = FLAG_FIELD[flag]; k, s = f"+ {flag}", BASE
    tb.append(dict(co=flag, truong=f, mo_ta=DESC[flag],
                   EM_train_baseline=A.loc[s, (f, "Train")], EM_train_chi_bat_co=A.loc[k, (f, "Train")],
                   EM_val_baseline=A.loc[s, (f, "Validation")], EM_val_chi_bat_co=A.loc[k, (f, "Validation")],
                   duoc_chon="có" if chosen[flag] else "không"))
tb = pd.DataFrame(tb); tb.to_csv(OUT_DIR / "improvement_explanations.csv", index=False, encoding="utf-8-sig"); display(tb)

README = f'''# Bàn giao người 4 -> người 5
Cấu hình đóng băng (chọn trên {SELECT_ON}): `outputs/improved/frozen_config.json`. Không chỉnh sau khi xem Validation/Test.
Kết quả: `improved_train.csv`, `improved_validation.csv`, `validation_and_train_comparison.csv`, `f1_company_address.csv`, `improvement_explanations.csv`, `ablation_tung_co.csv`.
## Cách dùng (người 5)
```python
from src.improved import ocr_frozen, extract_improved, norm_date, norm_total, norm_text, exact
text = ocr_frozen(pil_image)             # (nếu cần OCR từ ảnh) dùng đúng bước tiền xử lý đã đóng băng
pred = extract_improved(text)            # dict company/date/address/total (giá trị GỐC; không tìm thấy = None)
ok = exact(pred["total"], gold_total, "total")   # so khớp sau chuẩn hóa
```
Cấu hình đọc từ `outputs/improved/frozen_config.json`. Chạy trên Test đúng một lần, không chỉnh luật sau đó.
Nguồn OCR: {OCR_SOURCE}. Ghi chú: nếu mục 5 còn tắt (RUN_PREPROC=False) thì chưa thử tiền xử lý ảnh.

## Kết luận (Validation, n={len(va)})
{verdict}
'''
(OUT_DIR / "HUONG_DAN_NGUOI_5.md").write_text(README, encoding="utf-8"); print(README)

,co,truong,mo_ta,EM_train_baseline,EM_train_chi_bat_co,EM_val_baseline,EM_val_chi_bat_co,duoc_chon
0,clean_text,macro,Làm sạch ký tự rác/dòng ký hiệu (đo bằng macro),0.4341,0.4356,0.468,0.470,không
1,clean_date,date,"Sửa nhầm ký tự O/0, l/1, S/5 trong chuỗi ngày",0.6946,0.6966,0.752,0.752,có
2,date_v2,date,"Kiểm tra ngày hợp lệ, nhiều định dạng, ưu tiên...",0.6946,0.7425,0.752,0.808,có
3,clean_amount,total,Sửa nhầm ký tự trong số tiền,0.5329,0.5349,0.568,0.568,có
4,total_kw,total,"Thêm từ khóa total, không loại nhầm TOTAL INCL...",0.5329,0.6088,0.568,0.664,có
5,total_nospace,total,So khớp từ khóa bỏ khoảng trắng (NetTotal),0.5329,0.5329,0.568,0.576,không
6,total_gstcut,total,Bỏ phần GST/TAX SUMMARY khi tìm total,0.5329,0.5329,0.568,0.568,không
7,total_pos,total,Không có từ khóa → số lớn nhất ở phần dưới hóa...,0.5329,0.5768,0.568,0.600,có
8,company_v2,company,"Chấm điểm dòng tên công ty (hậu tố, chữ HOA, l...",0.2555,0.3453,0.248,0.360,có
9,addr_v2,address,"Dấu hiệu địa chỉ rộng hơn, cắt TEL/FAX cuối dòng",0.2535,0.2595,0.304,0.312,có


# Bàn giao người 4 -> người 5
Cấu hình đóng băng (chọn trên train): `outputs/improved/frozen_config.json`. Không chỉnh sau khi xem Validation/Test.
Kết quả: `improved_train.csv`, `improved_validation.csv`, `validation_and_train_comparison.csv`, `f1_company_address.csv`, `improvement_explanations.csv`, `ablation_tung_co.csv`.
## Cách dùng (người 5)
```python
from src.improved import ocr_frozen, extract_improved, norm_date, norm_total, norm_text, exact
text = ocr_frozen(pil_image)             # (nếu cần OCR từ ảnh) dùng đúng bước tiền xử lý đã đóng băng
pred = extract_improved(text)            # dict company/date/address/total (giá trị GỐC; không tìm thấy = None)
ok = exact(pred["total"], gold_total, "total")   # so khớp sau chuẩn hóa
```
Cấu hình đọc từ `outputs/improved/frozen_config.json`. Chạy trên Test đúng một lần, không chỉnh luật sau đó.
Nguồn OCR: original (gray+resize2x). Ghi chú: nếu mục 5 còn tắt (RUN_PREPROC=False) thì chưa thử tiền xử lý ảnh.

## Kết luận (Validation, n=125

## 12. Kiểm tra bàn giao: `src/improved.py` + cấu hình đóng băng cho đúng kết quả đã báo cáo
Gọi `extract_improved()` (hàm mà `demo.py` dùng) trên toàn bộ Validation và Train, rồi so với kết quả ở mục 9–11. Phải **không lệch ảnh nào**, nếu lệch thì dừng.

In [18]:
import importlib, src.improved as _mod
importlib.reload(_mod)          # đọc lại frozen_config.json vừa ghi ở mục 8
def _same(a, b): return all((("" if pd.isna(a[f]) else a[f]) == ("" if pd.isna(b[f]) else b[f])) for f in FIELDS)
bad = 0
for d, p in [(va, pv), (tr, pt)]:
    for t, rec in zip(d.ocr_text, p.to_dict("records")):
        out = _mod.extract_improved(t)
        assert set(out) == set(FIELDS), "extract_improved phải trả đúng 4 key"
        bad += not _same(out, rec)
assert bad == 0, f"extract_improved lệch kết quả đã báo cáo ở {bad} ảnh"
print("OK: extract_improved(src/improved.py) + frozen_config.json khớp 100% với kết quả mục 9-11 trên", len(va) + len(tr), "ảnh.")
print("Bàn giao: commit src/improved.py, outputs/improved/*.csv, frozen_config.json, notebook này.")

OK: extract_improved(src/improved.py) + frozen_config.json khớp 100% với kết quả mục 9-11 trên 626 ảnh.
Bàn giao: commit src/improved.py, outputs/improved/*.csv, frozen_config.json, notebook này.
